<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/01b_churn_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01b — Churn walkthrough: the class dataset

This notebook reproduces, step by step, the churn walkthrough shown in class (Module 6, Parts 1 and 2). Run it top to bottom: the numbers match the slides.

- **Part 1** builds the model: EDA, one engineered feature, encoding, a logistic regression, scores on a held-out test set.
- **Part 2** turns the probabilities into a decision: a threshold chosen from the cost of errors, and its value in euros.

Notebook 01 then goes further — precision-recall curves, calibration, lift — on its own synthetic data, so its numbers are different from these.

## Setup

In [ ]:
# Everything here is preinstalled on Colab. Running locally? pip install -r requirements.txt
import io, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve)
plt.rcParams['figure.dpi'] = 110
print('pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

## Load the class dataset

5,000 customers of an Italian telecom, 17 features and the target `churn` (0/1). The cell reads the CSV from the course repository; if the download fails, it looks for a local copy in `data/`.

In [ ]:
URL = 'https://raw.githubusercontent.com/paolobalasso/DataScienceCourse/main/data/churn_student_v2.csv'
LOCAL = [Path('churn_student_v2.csv'), Path('data/churn_student_v2.csv'), Path('../data/churn_student_v2.csv')]

def load_data():
    try:
        with urllib.request.urlopen(URL, timeout=20) as r:
            data = pd.read_csv(io.BytesIO(r.read()))
        print('Loaded from GitHub')
        return data
    except Exception as e:
        print(f'GitHub download failed ({type(e).__name__}); looking for a local copy')
    for p in LOCAL:
        if p.exists():
            print('Loaded from', p)
            return pd.read_csv(p)
    raise FileNotFoundError('churn_student_v2.csv not found: upload it next to this notebook')

df = load_data()
assert df.shape == (5000, 19), f'unexpected shape {df.shape}: download the current data/churn_student_v2.csv'
print(df.shape, '| churn rate', round(df['churn'].mean(), 3))
df.head()

## Part 1 · Build the model

### Step 1. EDA: distributions, correlations, missingness

In [ ]:
print('missing values:', int(df.isna().sum().sum()))
display(df.describe().round(2).T)
for col in ['contract_type', 'region', 'internet_service']:
    print(df.groupby(col)['churn'].agg(['mean', 'size']).round(3), '\n')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
df.groupby('contract_type')['churn'].mean().plot.bar(ax=axes[0], color='#1B263B', title='Churn rate by contract')
df.boxplot(column='monthly_charges', by='churn', ax=axes[1])
axes[1].set_title('Monthly charges by churn'); plt.suptitle(''); plt.tight_layout(); plt.show()

### Step 2. Engineer one feature

`tech_delta` = the best competitor's technology level minus ours. A positive gap means the competitor offers a better network where the customer lives.

In [ ]:
df['tech_delta'] = df['tech_level_best_competitor'] - df['tech_level_company']
print(df.groupby(pd.cut(df['tech_delta'], [-3, -1, -0.5, 0, 0.5, 1, 3]), observed=True)['churn'].mean().round(3))

### Step 3. Split, one-hot encode the categoricals, standardise the numerics

80% to train, 20% held out for testing, stratified on churn. The scaler learns its means and standard deviations on the training set only.

In [ ]:
CATS = ['contract_type', 'region', 'city_size', 'internet_service']
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['churn'])

def encode(d):
    return pd.get_dummies(d, columns=CATS, drop_first=True)

feat_cols = list(encode(df.drop(columns=['customer_id', 'churn'])).columns)
X_train = encode(train_df.drop(columns=['customer_id', 'churn']))[feat_cols].astype(float)
X_test = encode(test_df.drop(columns=['customer_id', 'churn']))[feat_cols].astype(float)
y_train, y_test = train_df['churn'].values, test_df['churn'].values

scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
print('train', X_train.shape, '| test', X_test.shape)

### Step 4. Fit a logistic regression

In [ ]:
model = LogisticRegression(max_iter=2000, C=1.0).fit(X_train_s, y_train)
p_test = model.predict_proba(X_test_s)[:, 1]          # predicted probability of churn

### Step 5. Score on the held-out 20%

The confusion matrix at τ = 0.5, the five metrics, the ROC curve against the random baseline.

In [ ]:
pred = (p_test >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f'Confusion matrix at 0.5: TN={tn} FP={fp} FN={fn} TP={tp}')
print(f'Accuracy {accuracy_score(y_test, pred):.3f} | Precision {precision_score(y_test, pred):.3f} | '
      f'Recall {recall_score(y_test, pred):.3f} | F1 {f1_score(y_test, pred):.3f} | '
      f'AUC {roc_auc_score(y_test, p_test):.3f}')

fpr, tpr, _ = roc_curve(y_test, p_test)
plt.figure(figsize=(4.8, 4.2))
plt.plot(fpr, tpr, color='#1B263B', lw=2, label=f'Logistic (AUC={roc_auc_score(y_test, p_test):.3f})')
plt.plot([0, 1], [0, 1], ls='--', color='grey', label='Random baseline')
plt.xlabel('False positive rate'); plt.ylabel('True positive rate'); plt.legend(); plt.show()

### The top-10 standardised coefficients

Contract type, tenure and monthly charges dominate; `tech_delta` follows. Look at the region dummies: on their own they add little, and their coefficients move once tenure, charges and contract are in the model. A region is a proxy, not a driver.

In [ ]:
coefs = pd.Series(model.coef_[0], index=feat_cols)
top10 = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(10)
print(top10.round(2))
top10.iloc[::-1].plot.barh(color=['#D4AF37' if v > 0 else '#1B263B' for v in top10.iloc[::-1]], figsize=(7, 4))
plt.axvline(0, color='black', lw=0.8); plt.title('Top-10 standardised coefficients'); plt.show()

# region as a proxy: churn by region, raw and within each contract type
print(df.pivot_table(index='region', columns='contract_type', values='churn', aggfunc='mean').round(3))

**Your turn (30 seconds):** write down one thing that surprised you.

## Part 2 · From probability to decision

Illustrative costs: a retention offer sent to a customer who would have stayed costs **C_FP = €30**; a missed churner costs **C_FN = €500** of margin. The closed form gives τ* = C_FP / (C_FP + C_FN).

In [ ]:
C_FP, C_FN = 30.0, 500.0
print(f'closed-form tau* = {C_FP / (C_FP + C_FN):.3f}')

thresholds = np.linspace(0.05, 0.95, 91)
rows = []
for t in thresholds:
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, (p_test >= t).astype(int)).ravel()
    rows.append({'tau': round(t, 2), 'FP': fp_t, 'FN': fn_t, 'TP': tp_t, 'cost': fp_t * C_FP + fn_t * C_FN})
sweep = pd.DataFrame(rows)
best = sweep.loc[sweep['cost'].idxmin()]
at_05 = sweep.loc[(sweep['tau'] - 0.5).abs().idxmin()]
print(f"empirical tau* = {best['tau']:.2f} | FP {int(best['FP'])} | FN {int(best['FN'])} | cost €{best['cost']:,.0f}")
print(f"cost at 0.5 = €{at_05['cost']:,.0f} | saving on the test set = €{at_05['cost'] - best['cost']:,.0f}")
print(f"customers contacted at tau*: {int(best['FP'] + best['TP'])} of {len(y_test)}")

plt.figure(figsize=(8, 3.8))
plt.plot(sweep['tau'], sweep['cost'], color='#1B263B', lw=2)
plt.axvline(best['tau'], color='green', ls='--', label=f"tau* = {best['tau']:.2f}")
plt.axvline(0.5, color='grey', ls=':', label='default 0.5')
plt.xlabel('threshold'); plt.ylabel('total cost (€)'); plt.legend(); plt.show()

Here the threshold is swept on the test set, to reproduce the slide. In a real project you choose it on a **validation** split and keep the test set for the final check: Notebook 01 does exactly that.

### Scaling to 100,000 customers

The test set has 1,000 customers: multiplying by 100 gives an order of magnitude, under the simple cost model above, not a budget figure. Before quoting a return, check the contact capacity of the retention team and the real save rate of the offer.

In [ ]:
saving = at_05['cost'] - best['cost']
print(f'saving on 1,000 test customers: €{saving:,.0f}  ->  scaled to 100,000: about €{saving * 100 / 1e6:.2f}M')

## Change one parameter

1. Set `C_FN = 300` and run Part 2 again: where does τ* go, and why?
2. Drop `tech_delta` from the features: what happens to the AUC and to the top coefficients?
3. Change `random_state` in the split: how much do the euro figures move?

**Next:** Notebook 01 — precision-recall curves, calibration and lift.